# Device Vector Addition (CUDA)

This notebook demonstrates **vector addition on the GPU** (the "device" in CUDA terminology).

We compute: **C = A + B**, where A, B, and C are arrays of floating-point numbers.

The workflow follows the typical CUDA pattern:
1. Allocate memory on the device
2. Copy input data from host to device
3. Launch the kernel
4. Copy the result back from device to host
5. Free device memory

In [1]:
!apt update -qq && apt install -y -qq nvidia-cuda-toolkit

144 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
The following additional packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libaccinj64-11.5 libatk-bridge2.0-0 libatk-wrapper-java
  libatk-wrapper-java-jni libatk1.0-0 libatk1.0-data libatspi2.0-0
  libbabeltrace1 libboost-regex1.74.0 libcub-dev libcublas11 libcublaslt11
  libcudart11.0 libcufft10 libcufftw10 libcuinj64-11.5 libcupti-dev
  libcupti-doc libcupti11.5 libcurand10 libcusolver11 libcusolvermg11
  libcusparse11 libdebuginfod-common libdebuginfod1 libegl-dev libgail-common
  libgail18 libgl-dev libgl1-mesa-dev libgles-dev libgles1 libglvnd-core-dev
  libglvnd-dev libglx-dev libgtk2.0-0 libgtk2.0-bin libgtk2.0-common libipt2
  libnppc11 libnppial11 libnppicc11 lib

## The Kernel and `vecAdd` Function

- **`vecAddKernel`** — a CUDA kernel that runs on the GPU. Each thread computes one element: `C[i] = A[i] + B[i]`. The boundary check `if (i < n)` prevents out-of-bounds access when the total number of threads exceeds `n`.
- **`vecAdd`** — the host function that orchestrates the GPU computation: allocates device memory, copies data host→device, launches the kernel, copies the result device→host, and frees device memory.
- **`ceil(n/256.0)`** — calculates the number of thread blocks needed so that every element is covered, with 256 threads per block.

In [2]:
%%bash
cat > vecadd_device.cu <<'EOF'
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#define CUDA_CHECK(call)                                                       \
    do {                                                                       \
        cudaError_t err = (call);                                              \
        if (err != cudaSuccess) {                                              \
            fprintf(stderr, "CUDA error at %s:%d - %s\n", __FILE__, __LINE__, \
                    cudaGetErrorString(err));                                   \
            exit(EXIT_FAILURE);                                                \
        }                                                                      \
    } while (0)

__global__ void vecAddKernel(float* A, float* B, float* C, int n) {
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    if (i < n) {
        C[i] = A[i] + B[i];
    }
}

void vecAdd(float* A_h, float* B_h, float* C_h, int n) {
    int size = n * sizeof(float);
    float *A_d, *B_d, *C_d;

    // Allocate memory on the device
    CUDA_CHECK(cudaMalloc((void**)&A_d, size));
    CUDA_CHECK(cudaMalloc((void**)&B_d, size));
    CUDA_CHECK(cudaMalloc((void**)&C_d, size));

    // Copy data from host to device
    CUDA_CHECK(cudaMemcpy(A_d, A_h, size, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(B_d, B_h, size, cudaMemcpyHostToDevice));

    // Launch the kernel
    vecAddKernel<<<ceil(n / 256.0), 256>>>(A_d, B_d, C_d, n);

    // Copy result from device to host
    CUDA_CHECK(cudaMemcpy(C_h, C_d, size, cudaMemcpyDeviceToHost));

    // Free memory on the device
    CUDA_CHECK(cudaFree(A_d));
    CUDA_CHECK(cudaFree(B_d));
    CUDA_CHECK(cudaFree(C_d));
}

int main() {
    int N = 1000;
    float* A = new float[N];
    float* B = new float[N];
    float* C = new float[N];

    srand(42);
    for (int i = 0; i < N; ++i) A[i] = static_cast<float>(rand()) / RAND_MAX;
    for (int i = 0; i < N; ++i) B[i] = static_cast<float>(rand()) / RAND_MAX;

    vecAdd(A, B, C, N);

    printf("Result C = A + B (first 10 elements):\n");
    for (int i = 0; i < 10; ++i) {
        printf("C[%d] = %f\n", i, C[i]);
    }

    delete[] A;
    delete[] B;
    delete[] C;

    return 0;
}
EOF

In [3]:
!nvcc vecadd_device.cu -o vecadd_device && ./vecadd_device

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Result C = A + B (first 10 elements):
C[0] = 0.947587
C[1] = 0.344192
C[2] = 1.136735
C[3] = 0.866441
C[4] = 0.368441
C[5] = 1.066958
C[6] = 0.938131
C[7] = 0.925742
C[8] = 0.745824
C[9] = 0.770067


## Code Walkthrough

### Error-checking macro

```c
#define CUDA_CHECK(call)                                                       \
    do {                                                                       \
        cudaError_t err = (call);                                              \
        if (err != cudaSuccess) {                                              \
            fprintf(stderr, "CUDA error at %s:%d - %s\n", __FILE__, __LINE__, \
                    cudaGetErrorString(err));                                   \
            exit(EXIT_FAILURE);                                                \
        }                                                                      \
    } while (0)
```

Every CUDA API call returns a `cudaError_t`. This macro wraps each call, checks the return value, and aborts with a descriptive message if anything goes wrong. Without this, CUDA errors can silently corrupt results.

### The kernel

```c
__global__ void vecAddKernel(float* A, float* B, float* C, int n) {
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    if (i < n) {
        C[i] = A[i] + B[i];
    }
}
```

- **`__global__`** — marks this as a kernel: called from the host (CPU), executed on the device (GPU).
- **`int i = blockDim.x * blockIdx.x + threadIdx.x`** — computes a globally unique thread index. `blockDim.x` is threads per block (256), `blockIdx.x` is which block this thread belongs to, and `threadIdx.x` is the thread's position within that block.
- **`if (i < n)`** — boundary check. We launch `ceil(n/256)` blocks × 256 threads, which may exceed `n`. This guard prevents out-of-bounds memory access.
- **`C[i] = A[i] + B[i]`** — each thread computes exactly one element of the output.

### The host function

```c
void vecAdd(float* A_h, float* B_h, float* C_h, int n) { ... }
```

This function orchestrates the five-step CUDA workflow:

| Step | API Call | Purpose |
|------|----------|---------|
| 1 | `cudaMalloc` | Allocate device memory for A, B, and C |
| 2 | `cudaMemcpy(..., HostToDevice)` | Copy input vectors A and B to the GPU |
| 3 | `vecAddKernel<<<blocks, 256>>>` | Launch the kernel on the GPU |
| 4 | `cudaMemcpy(..., DeviceToHost)` | Copy the result C back to the CPU |
| 5 | `cudaFree` | Free the device memory |

### Kernel launch configuration

```c
vecAddKernel<<<ceil(n / 256.0), 256>>>(A_d, B_d, C_d, n);
```

- **First argument** `ceil(n / 256.0)` — number of blocks in the grid. For `n = 1000`, this gives `ceil(1000/256) = 4` blocks.
- **Second argument** `256` — number of threads per block.
- Total threads launched: `4 × 256 = 1024`, which covers all 1000 elements (the `if (i < n)` check handles the extra 24 threads).

## Why No For Loop in the Kernel?

On the CPU, vector addition requires a for loop — one thread iterates through every element sequentially:

```c
// CPU version
for (int i = 0; i < n; i++) {
    C[i] = A[i] + B[i];
}
```

The CUDA kernel has **no loop at all**:

```c
// GPU kernel
int i = blockDim.x * blockIdx.x + threadIdx.x;
if (i < n) {
    C[i] = A[i] + B[i];
}
```

**The GPU's threading model replaces the loop.** Here's why:

1. **One thread per element** — The kernel launch `<<<ceil(n/256.0), 256>>>` spawns at least `n` threads. Each thread gets a unique index `i` from its block and thread IDs, so every element is assigned to exactly one thread.

2. **The launch configuration *is* the loop** — The for loop's job of "do this for every index" is replaced by "launch a thread for every index." The grid dimensions define the iteration space.

3. **All threads run in parallel** — Instead of one thread doing `n` iterations sequentially, `n` threads each do exactly **one** computation simultaneously.

| | CPU | GPU |
|---|---|---|
| **Model** | 1 thread, `n` iterations | `n` threads, 1 computation each |
| **Iteration** | `for` loop | Thread index mapping |
| **Execution** | Sequential | Massively parallel |

**In short:** The CPU for loop says *"one thread, repeat n times."* The CUDA kernel says *"n threads, each do it once."*